# Latent-TLG sweep — results

Visualizes the per-dataset latent-TLG GIC sweep (`make tlg-all-latent-gic`). Reads the
per-network JSON cache directly, so it works **while the sweep is still running** (partial)
and on the finished run — just re-run the cells as more networks complete.

- **Mean KL rank of each family across datasets** (lower = better; 1 = best fit).
- **Selected graphs per family** — for each model family, example graphs where it ranks #1,
  with the real vs. generated structure (clustering, assortativity, avg degree).

In [ ]:
import json, glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

FAMILIES = ["TLG", "ER", "BA", "WS", "KR", "GRG", "SBM"]
CB = {"TLG":"#0072B2","ER":"#E69F00","BA":"#009E73","WS":"#CC79A7",
      "KR":"#D55E00","GRG":"#56B4E9","SBM":"#000000"}

def find_runs_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / "scripts" / "closedform" / "runs"
        if cand.exists():
            return cand
    raise RuntimeError("could not locate scripts/closedform/runs")

RUNS = find_runs_root()
print("runs root:", RUNS)

In [ ]:
# Load every cached network into a tidy (network x family) DataFrame
rows = []
for jp in glob.glob(str(RUNS / "tlg_latent_*_gic" / "cache" / "*.json")):
    try:
        d = json.loads(Path(jp).read_text())
    except Exception:
        continue
    if "families" not in d:
        continue
    fam = pd.DataFrame(d["families"])
    fam["kl_rank"] = fam["kl"].rank(method="min").astype(int)
    real = d.get("real", {})
    for _, r in fam.iterrows():
        rows.append(dict(
            dataset=d["dataset"], graph=str(d["id"]),
            real_nodes=real.get("nodes"), real_edges=real.get("edges"),
            real_clustering=real.get("clustering"),
            real_assortativity=real.get("assortativity"),
            real_avg_degree=real.get("avg_degree"),
            model=r["model"], kl=r["kl"], gic=r["gic"], n_params=r["n_params"],
            kl_rank=int(r["kl_rank"]),
            gen_edges=r.get("edges"), gen_avg_degree=r.get("avg_degree"),
            gen_clustering=r.get("clustering"), gen_assortativity=r.get("assortativity"),
        ))
df = pd.DataFrame(rows)
n_graphs = df.groupby("dataset")["graph"].nunique()
print(f"loaded {df['graph'].nunique()} networks across {df['dataset'].nunique()} datasets")
print(n_graphs.to_string())
df.head()

## Mean KL rank of families across datasets

Per dataset, every network ranks the 7 families by raw KL (1 = best). The table is the
mean rank per family per dataset; the bar chart is the **overall** mean rank
macro-averaged across datasets (each dataset weighted equally, so the large datasets
don't dominate).

In [ ]:
# mean KL rank per (dataset, family) and overall macro-average
per_ds = (df.groupby(["dataset","model"])["kl_rank"].mean()
            .unstack("model")[FAMILIES])
overall = per_ds.mean(axis=0).sort_values()
winrate = (df.assign(win=(df.kl_rank==1).astype(float))
             .groupby(["dataset","model"])["win"].mean().unstack("model")[FAMILIES])

print("Overall mean KL rank (macro-averaged across datasets, lower=better):")
print(overall.round(3).to_string())
print("\nMean KL rank per dataset:")
display(per_ds[overall.index].round(2))
print("Win rate (fraction of a dataset's graphs where the family ranks #1):")
display((winrate[overall.index]*100).round(1))

In [ ]:
order = list(overall.index)
fig, ax = plt.subplots(1, 3, figsize=(20, 5.2))

# (1) overall mean rank
ax[0].bar(order, overall[order].values, color=[CB[m] for m in order])
ax[0].set_title("Overall mean KL rank (macro-avg across datasets)")
ax[0].set_ylabel("mean KL rank (1 = best)"); ax[0].grid(alpha=.25, axis="y")
for i,m in enumerate(order): ax[0].text(i, overall[m], f"{overall[m]:.2f}", ha="center", va="bottom")

# (2) per-dataset heatmap
H = per_ds[order].T
im = ax[1].imshow(H.values, aspect="auto", cmap="RdYlGn_r", vmin=1, vmax=7)
ax[1].set_xticks(range(H.shape[1])); ax[1].set_xticklabels(H.columns, rotation=45, ha="right")
ax[1].set_yticks(range(len(order))); ax[1].set_yticklabels(order)
ax[1].set_title("Mean KL rank per dataset")
for yi in range(len(order)):
    for xi in range(H.shape[1]):
        v = H.values[yi, xi]
        if np.isfinite(v): ax[1].text(xi, yi, f"{v:.1f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax[1], shrink=.8, label="mean KL rank")

# (3) TLG win-rate per dataset
tlg_wr = (winrate["TLG"]*100).sort_values(ascending=False)
ax[2].bar(tlg_wr.index, tlg_wr.values, color=CB["TLG"])
ax[2].set_title("TLG win rate (% graphs where TLG is #1)")
ax[2].set_ylabel("% ranked #1"); ax[2].set_ylim(0,100)
ax[2].set_xticks(range(len(tlg_wr))); ax[2].set_xticklabels(tlg_wr.index, rotation=45, ha="right"); ax[2].grid(alpha=.25, axis="y")
plt.tight_layout(); plt.show()

## Selected graphs per family

For each family, a few graphs where it ranks **#1** on raw KL (its best fits), shown with
the **real vs. generated** structure. This is where you see *what each family captures*:
e.g. GRG matches spatial/clustered graphs, SBM dense communities, and TLG reproduces both
the degree and the clustering/community structure.

In [ ]:
metric_cols = ["dataset","graph","real_nodes","kl",
               "real_clustering","gen_clustering",
               "real_assortativity","gen_assortativity",
               "real_avg_degree","gen_avg_degree"]
for fam in FAMILIES:
    sub = df[df.model==fam]
    wins = sub[sub.kl_rank==1].sort_values("kl")
    print(f"\n=== {fam}: ranked #1 on {len(wins)}/{sub['graph'].nunique()} graphs "
          f"(showing up to 3 best-KL examples) ===")
    if len(wins):
        display(wins[metric_cols].head(3).round(3).reset_index(drop=True))
    else:
        print("  (no #1 finishes yet)")

### How well does each family reproduce structure?

Real vs. generated **clustering** and **assortativity** for TLG and SBM across all fitted
graphs (points on the dashed y=x line = perfect structural match).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for j,(metric,real_c,gen_c) in enumerate([
        ("clustering","real_clustering","gen_clustering"),
        ("assortativity","real_assortativity","gen_assortativity")]):
    for fam in ["TLG","SBM","GRG"]:
        s = df[(df.model==fam)].dropna(subset=[real_c,gen_c])
        ax[j].scatter(s[real_c], s[gen_c], s=12, alpha=.4, color=CB[fam], label=fam)
    lo = min(df[real_c].min(), df[gen_c].min())
    hi = max(df[real_c].max(), df[gen_c].max())
    ax[j].plot([lo,hi],[lo,hi], "k--", lw=1, alpha=.6)
    ax[j].set_xlabel(f"real {metric}"); ax[j].set_ylabel(f"generated {metric}")
    ax[j].set_title(f"{metric}: real vs generated"); ax[j].legend(); ax[j].grid(alpha=.25)
plt.tight_layout(); plt.show()